In [1]:
from rdkit import Chem

smiles = "CCO"
mol = Chem.MolFromSmiles(smiles)

print("Number of atoms:", mol.GetNumAtoms())
print("Number of bonds:", mol.GetNumBonds())

for atom in mol.GetAtoms():
    print(
        "Atom index:", atom.GetIdx(),
        "Element:", atom.GetSymbol(),
        "Atomic number:", atom.GetAtomicNum(),
        "Degree:", atom.GetDegree(),
        "Formal charge:", atom.GetFormalCharge(),
        "Aromatic:", atom.GetIsAromatic()
    )

print("\nBonds:")

for bond in mol.GetBonds():
    print(
        bond.GetBeginAtomIdx(),
        "<->",
        bond.GetEndAtomIdx(),
        "Type:", bond.GetBondType()
    )

Number of atoms: 3
Number of bonds: 2
Atom index: 0 Element: C Atomic number: 6 Degree: 1 Formal charge: 0 Aromatic: False
Atom index: 1 Element: C Atomic number: 6 Degree: 2 Formal charge: 0 Aromatic: False
Atom index: 2 Element: O Atomic number: 8 Degree: 1 Formal charge: 0 Aromatic: False

Bonds:
0 <-> 1 Type: SINGLE
1 <-> 2 Type: SINGLE


In [2]:
import torch

atom_features = []

for atom in mol.GetAtoms():
    features = [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic())
    ]
    atom_features.append(features)

x = torch.tensor(atom_features, dtype=torch.float32)

print(x)
print("Shape:", x.shape)

tensor([[6., 1., 0., 0.],
        [6., 2., 0., 0.],
        [8., 1., 0., 0.]])
Shape: torch.Size([3, 4])


In [3]:
edge_list = []
edge_features = []

for bond in mol.GetBonds():
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()

    bond_type = bond.GetBondType()

    features = [
        int(bond_type == Chem.BondType.SINGLE),
        int(bond_type == Chem.BondType.DOUBLE),
        int(bond_type == Chem.BondType.TRIPLE),
        int(bond_type == Chem.BondType.AROMATIC)
    ]

    # i -> j
    edge_list.append([i, j])
    edge_features.append(features)

    # j -> i
    edge_list.append([j, i])
    edge_features.append(features)

edge_index = torch.tensor(
    edge_list, dtype=torch.long
).t().contiguous()

edge_attr = torch.tensor(
    edge_features, dtype=torch.float32
)

print("edge_index:")
print(edge_index)

print("\nedge_attr:")
print(edge_attr)

print("\nedge_index shape:", edge_index.shape)
print("edge_attr shape:", edge_attr.shape)

edge_index:
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])

edge_attr:
tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]])

edge_index shape: torch.Size([2, 4])
edge_attr shape: torch.Size([4, 4])


In [ ]:
!pip install torch_geometric
import torch_geometric
print(torch_geometric.__version__)

In [5]:
from torch_geometric.data import Data

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr
)

print(data)
print("Number of nodes:", data.num_nodes)
print("Number of directed edges:", data.num_edges)
print("Node features:", data.num_node_features)
print("Edge features:", data.num_edge_features)

Data(x=[3, 4], edge_index=[2, 4], edge_attr=[4, 4])
Number of nodes: 3
Number of directed edges: 4
Node features: 4
Edge features: 4


In [6]:
data.y = torch.tensor([6.5], dtype=torch.float32)

print(data)
print("Target:", data.y)


Data(x=[3, 4], edge_index=[2, 4], edge_attr=[4, 4], y=[1])
Target: tensor([6.5000])


In [7]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

class SimpleGNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(
            in_channels=4,
            out_channels=8
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        return x

gnn = SimpleGNN()

print(gnn)

SimpleGNN(
  (conv1): GCNConv(4, 8)
)


In [8]:
gnn.eval()

with torch.no_grad():
    h = gnn(data.x, data.edge_index)

print(h)
print("Input shape :", data.x.shape)
print("Output shape:", h.shape)

tensor([[-1.0611,  3.0714, -2.5959, -2.7440, -2.5107, -1.8087, -0.8549, -2.7191],
        [-1.4447,  4.2124, -3.9371, -3.8040, -3.5212, -2.5345, -1.4458, -3.7826],
        [-1.2193,  3.5486, -3.2383, -3.1961, -2.9502, -2.1239, -1.1612, -3.1754]])
Input shape : torch.Size([3, 4])
Output shape: torch.Size([3, 8])


In [9]:
from torch_geometric.nn import global_mean_pool

# All three atoms belong to molecule 0
batch = torch.tensor([0, 0, 0], dtype=torch.long)

h_mol = global_mean_pool(h, batch)

print(h_mol)
print("Atom embedding shape    :", h.shape)
print("Molecule embedding shape:", h_mol.shape)

tensor([[-1.2417,  3.6108, -3.2571, -3.2480, -2.9940, -2.1557, -1.1540, -3.2257]])
Atom embedding shape    : torch.Size([3, 8])
Molecule embedding shape: torch.Size([1, 8])


In [10]:
prediction_layer = nn.Linear(8, 1)

y_pred = prediction_layer(h_mol)

print(y_pred)
print("Molecular embedding:", h_mol.shape)
print("Prediction shape   :", y_pred.shape)

tensor([[-3.0780]], grad_fn=<AddmmBackward0>)
Molecular embedding: torch.Size([1, 8])
Prediction shape   : torch.Size([1, 1])


In [11]:
#Complete Workflow
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, global_mean_pool


class MolecularGNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(4, 32)
        self.conv2 = GCNConv(32, 32)

        self.fc = nn.Linear(32, 1)

    def forward(self, x, edge_index, batch):

        # Message-passing layer 1
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # Message-passing layer 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Atom embeddings -> molecular embedding
        x = global_mean_pool(x, batch)

        # Molecular embedding -> pIC50
        x = self.fc(x)

        return x

In [12]:
model = MolecularGNN()

batch = torch.zeros(
    data.num_nodes,
    dtype=torch.long
)

with torch.no_grad():
    prediction = model(
        data.x,
        data.edge_index,
        batch
    )

print(model)
print("Prediction:", prediction)
print("Prediction shape:", prediction.shape)

MolecularGNN(
  (conv1): GCNConv(4, 32)
  (conv2): GCNConv(32, 32)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Prediction: tensor([[0.4807]])
Prediction shape: torch.Size([1, 1])


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GINEConv, global_mean_pool


class BondAwareGNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Raw features -> learned 32-dimensional representations
        self.node_encoder = nn.Linear(4, 32)
        self.edge_encoder = nn.Linear(4, 32)

        # Neural network used inside GINE message passing
        mlp1 = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        self.conv1 = GINEConv(mlp1)

        self.fc = nn.Linear(32, 1)

    def forward(self, x, edge_index, edge_attr, batch):

        x = self.node_encoder(x)
        edge_attr = self.edge_encoder(edge_attr)

        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)

        x = global_mean_pool(x, batch)

        return self.fc(x)

In [14]:
model = BondAwareGNN()

batch = torch.zeros(data.num_nodes, dtype=torch.long)

with torch.no_grad():
    prediction = model(
        data.x,
        data.edge_index,
        data.edge_attr,
        batch
    )

print(model)
print("Prediction:", prediction)
print("Prediction shape:", prediction.shape)

BondAwareGNN(
  (node_encoder): Linear(in_features=4, out_features=32, bias=True)
  (edge_encoder): Linear(in_features=4, out_features=32, bias=True)
  (conv1): GINEConv(nn=Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
  ))
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Prediction: tensor([[-0.6495]])
Prediction shape: torch.Size([1, 1])


In [15]:
# Original: all ethanol bonds are single
edge_single = data.edge_attr.clone()

# Make a second version
edge_modified = data.edge_attr.clone()

# Change the C0 <-> C1 bond from SINGLE to DOUBLE
# Directed edges 0 and 1 correspond to 0->1 and 1->0
edge_modified[0] = torch.tensor([0., 1., 0., 0.])
edge_modified[1] = torch.tensor([0., 1., 0., 0.])

print("Original:")
print(edge_single)

print("\nModified:")
print(edge_modified)

Original:
tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]])

Modified:
tensor([[0., 1., 0., 0.],
        [0., 1., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]])


In [16]:
model.eval()

with torch.no_grad():

    pred_single = model(
        data.x,
        data.edge_index,
        edge_single,
        batch
    )

    pred_modified = model(
        data.x,
        data.edge_index,
        edge_modified,
        batch
    )

print("Original bond prediction :", pred_single.item())
print("Modified bond prediction :", pred_modified.item())

Original bond prediction : -0.6495009064674377
Modified bond prediction : -0.6255460381507874


In [17]:
def one_hot(value, choices):
    return [int(value == choice) for choice in choices]

In [18]:
elements = ["C", "N", "O", "S", "F", "Cl", "Br", "P"]

print(one_hot("C", elements))
print(one_hot("O", elements))
print(one_hot("Cl", elements))

[1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, 0, 0]


In [19]:
elements = [
    "C", "N", "O", "S",
    "F", "Cl", "Br", "P",
    "Other"
]

def one_hot_with_unknown(value, choices):
    if value not in choices[:-1]:
        value = "Other"

    return [int(value == choice) for choice in choices]

In [20]:
print(one_hot_with_unknown("C", elements))
print(one_hot_with_unknown("I", elements))
print(one_hot_with_unknown("Si", elements))

[1, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 1]


In [21]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

df = pd.read_csv("ache_ic50_clean.csv")
df = df[np.isfinite(df["pIC50"])].copy()
df = df.reset_index(drop=True)


In [22]:
from collections import Counter
from rdkit import Chem

element_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        for atom in mol.GetAtoms():
            element_counts[atom.GetSymbol()] += 1

print("Elements found:")
for element, count in element_counts.most_common():
    print(f"{element:>3s}: {count}")

Elements found:
  C: 149695
  O: 16710
  N: 16697
 Cl: 1733
  F: 1325
  S: 964
 Br: 681
  I: 109
  P: 53
 Se: 33
  B: 21
 Si: 6
  H: 4
 Na: 1


In [23]:
elements = [
    "C", "N", "O", "F", "P", "S",
    "Cl", "Br", "I",
    "B", "Si", "Se", "Na",
    "Other"
]

In [24]:
degree_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        for atom in mol.GetAtoms():
            degree_counts[atom.GetDegree()] += 1

print("Degrees found:")
for degree, count in sorted(degree_counts.items()):
    print(f"Degree {degree}: {count}")

degrees = [0, 1, 2, 3, 4, "Other"]    

Degrees found:
Degree 0: 1234
Degree 1: 24690
Degree 2: 104092
Degree 3: 55622
Degree 4: 2394


In [25]:
charge_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        for atom in mol.GetAtoms():
            charge_counts[atom.GetFormalCharge()] += 1

print("Formal charges found:")
for charge, count in sorted(charge_counts.items()):
    print(f"Charge {charge:+d}: {count}")

formal_charges = [-1, 0, 1, "Other"]    

Formal charges found:
Charge -1: 1134
Charge +0: 185580
Charge +1: 1318


In [26]:
#Hybridization
hybrid_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        for atom in mol.GetAtoms():
            hybrid_counts[str(atom.GetHybridization())] += 1

print("Hybridizations found:")
for hybrid, count in hybrid_counts.most_common():
    print(f"{hybrid}: {count}")

Hybridizations found:
SP2: 121428
SP3: 65835
SP: 762
S: 5
SP3D: 2


In [27]:
hybridizations = [
    Chem.HybridizationType.SP,
    Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3,
    "Other"
]

In [28]:
#Number of Implicit H-bonds
h_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        for atom in mol.GetAtoms():
            h_counts[atom.GetTotalNumHs()] += 1

print("Total H counts:")
for h, count in sorted(h_counts.items()):
    print(f"H count {h}: {count}")

Total H counts:
H count 0: 76501
H count 1: 61368
H count 2: 38757
H count 3: 11406


In [29]:
hydrogen_counts = [0, 1, 2, 3, "Other"]

In [30]:
#A
aromatic_ring_counts = Counter()

for smi in df["canonical_smiles"]:
    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        ring_info = mol.GetRingInfo()
        aromatic_rings = 0

        for ring in ring_info.AtomRings():
            # A ring is counted as aromatic if all atoms in it are aromatic
            if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
                aromatic_rings += 1

        aromatic_ring_counts[aromatic_rings] += 1

print("Aromatic rings per molecule:")
for n_rings, count in sorted(aromatic_ring_counts.items()):
    print(f"{n_rings} aromatic rings: {count} molecules")

Aromatic rings per molecule:
0 aromatic rings: 184 molecules
1 aromatic rings: 857 molecules
2 aromatic rings: 1950 molecules
3 aromatic rings: 1888 molecules
4 aromatic rings: 1083 molecules
5 aromatic rings: 275 molecules
6 aromatic rings: 40 molecules
7 aromatic rings: 8 molecules
8 aromatic rings: 1 molecules
9 aromatic rings: 1 molecules


In [31]:
for atom in mol.GetAtoms():
    aromatic = int(atom.GetIsAromatic())

    print(
        atom.GetSymbol(),
        "Aromatic:", aromatic
    )

C Aromatic: 0
N Aromatic: 0
C Aromatic: 0
C Aromatic: 0
C Aromatic: 0
C Aromatic: 0
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
O Aromatic: 0
C Aromatic: 0
O Aromatic: 0
N Aromatic: 0
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
N Aromatic: 0
C Aromatic: 0
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 1
C Aromatic: 0


In [32]:
#\[14_{\rm element} +6_{\rm degree} +4_{\rm charge} +4_{\rm hybridization} + 1 (IsAromatic) +5H=29\]
features = (
    one_hot_with_unknown(atom.GetSymbol(), elements)
    + one_hot_with_unknown(atom.GetDegree(), degrees)
    + one_hot_with_unknown(atom.GetFormalCharge(), formal_charges)
    + one_hot_with_unknown(atom.GetHybridization(), hybridizations)
    + [int(atom.GetIsAromatic())]
    + one_hot_with_unknown(atom.GetTotalNumHs(), hydrogen_counts)
)

In [33]:
elements = [
    "C", "N", "O", "F", "P", "S",
    "Cl", "Br", "I", "B", "Si", "Se", "Na",
    "Other"
]

degrees = [0, 1, 2, 3, 4, "Other"]

formal_charges = [-1, 0, 1, "Other"]

hybridizations = [
    Chem.HybridizationType.SP,
    Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3,
    "Other"
]

hydrogen_counts = [0, 1, 2, 3, "Other"]

In [34]:
def one_hot_with_unknown(value, choices):
    known_choices = choices[:-1]

    if value not in known_choices:
        value = "Other"

    return [int(value == choice) for choice in choices]

In [35]:
def get_atom_features(atom):

    features = (
        one_hot_with_unknown(
            atom.GetSymbol(),
            elements
        )
        +
        one_hot_with_unknown(
            atom.GetDegree(),
            degrees
        )
        +
        one_hot_with_unknown(
            atom.GetFormalCharge(),
            formal_charges
        )
        +
        one_hot_with_unknown(
            atom.GetHybridization(),
            hybridizations
        )
        +
        [int(atom.GetIsAromatic())]
        +
        one_hot_with_unknown(
            atom.GetTotalNumHs(),
            hydrogen_counts
        )
    )

    return features

In [36]:
mol = Chem.MolFromSmiles("CCO")

atom = mol.GetAtomWithIdx(0)

features = get_atom_features(atom)

print(features)
print("Number of features:", len(features))

[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]
Number of features: 34


In [37]:
bond_types = [
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
    "Other"
]

def get_bond_features(bond):

    features = (
        one_hot_with_unknown(
            bond.GetBondType(),
            bond_types
        )
        +
        [int(bond.GetIsConjugated())]
        +
        [int(bond.IsInRing())]
    )

    return features

In [38]:
mol = Chem.MolFromSmiles("CCO")

for bond in mol.GetBonds():

    features = get_bond_features(bond)

    print(
        bond.GetBeginAtomIdx(),
        "->",
        bond.GetEndAtomIdx(),
        features,
        "length =", len(features)
    )

0 -> 1 [1, 0, 0, 0, 0, 0, 0] length = 7
1 -> 2 [1, 0, 0, 0, 0, 0, 0] length = 7


In [39]:
from torch_geometric.data import Data

def smiles_to_graph(smiles, target=None):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    # -------------------------
    # Node features
    # -------------------------
    node_features = []

    for atom in mol.GetAtoms():
        node_features.append(
            get_atom_features(atom)
        )

    x = torch.tensor(
        node_features,
        dtype=torch.float32
    )

    # -------------------------
    # Edges + bond features
    # -------------------------
    edge_list = []
    edge_features = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        features = get_bond_features(bond)

        # i -> j
        edge_list.append([i, j])
        edge_features.append(features)

        # j -> i
        edge_list.append([j, i])
        edge_features.append(features)

    edge_index = torch.tensor(
        edge_list,
        dtype=torch.long
    ).t().contiguous()

    edge_attr = torch.tensor(
        edge_features,
        dtype=torch.float32
    )

    # -------------------------
    # PyG graph
    # -------------------------
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    if target is not None:
        data.y = torch.tensor(
            [target],
            dtype=torch.float32
        )

    return data

In [40]:
row = df.iloc[0]

graph = smiles_to_graph(
    row["canonical_smiles"],
    row["pIC50"]
)

print("SMILES:", row["canonical_smiles"])
print("pIC50 :", row["pIC50"])

print(graph)

print("Node tensor shape :", graph.x.shape)
print("Edge index shape  :", graph.edge_index.shape)
print("Edge attr shape   :", graph.edge_attr.shape)
print("Target shape      :", graph.y.shape)

SMILES: CN(CCOCCNC(=S)NC(=O)c1ccccc1)Cc1ccccc1
pIC50 : 6.920818753952375
Data(x=[26, 34], edge_index=[2, 54], edge_attr=[54, 7], y=[1])
Node tensor shape : torch.Size([26, 34])
Edge index shape  : torch.Size([2, 54])
Edge attr shape   : torch.Size([54, 7])
Target shape      : torch.Size([1])
